# Baseline -> Mining -> GRPO -> Eval (Colab)

This notebook runs the full baseline RL pipeline:
1. Mine RL training data with the baseline model
2. Train GRPO from the baseline model
3. Evaluate the resulting GRPO checkpoint

In [ ]:
!pip -q install -U pip
!pip -q install -U torch transformers datasets peft trl accelerate bitsandbytes pyyaml tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_DIR = '/content/Olympiad-AI'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/laureneproctor/Olympiad-AI.git /content/Olympiad-AI

os.chdir(REPO_DIR)
print('cwd:', os.getcwd())

In [ ]:
import copy
import json
import yaml

# ------------------------------
# USER CONFIG
# ------------------------------
MODEL_KEY = 'deepseekmath'  # 'deepseekmath' or 'qwen'
EXPERIMENT_NAME = 'exp2'

DATASET_PATH = '/content/drive/MyDrive/Math Olympiad Competition/Experimentation/processed_splits/omr_aimo3/splits/100000'
DATASET_SPLIT_FOR_MINING = 'val2'
EVAL_SPLIT = 'test'

MAX_MINE_ITEMS = 250
EVAL_MAX_ITEMS = 200
EVAL_MAJN_ITEMS = 200

BASE_ROOT = '/content/drive/MyDrive/Math Olympiad Competition/Experimentation'
MODELS_ROOT = f'{BASE_ROOT}/models'
RL_TRAIN_ROOT = f'{BASE_ROOT}/rl_train'
RUNS_ROOT = f'{BASE_ROOT}/runs'

mine_cfg_map = {
    'deepseekmath': '/content/Olympiad-AI/aimo3/configs/mine_baseline_deepseekmath.yaml',
    'qwen': '/content/Olympiad-AI/aimo3/configs/mine_baseline_qwen.yaml',
}
grpo_cfg_map = {
    'deepseekmath': '/content/Olympiad-AI/aimo3/configs/grpo_baseline_deepseekmath.yaml',
    'qwen': '/content/Olympiad-AI/aimo3/configs/grpo_baseline_qwen.yaml',
}
eval_cfg_map = {
    'deepseekmath': '/content/Olympiad-AI/aimo3/configs/evaluate_sft_deepseekmath.yaml',
    'qwen': '/content/Olympiad-AI/aimo3/configs/evaluate_sft_qwen.yaml',
}
sft_cfg_map = {
    'deepseekmath': '/content/Olympiad-AI/aimo3/configs/sft_deepseekmath.yaml',
    'qwen': '/content/Olympiad-AI/aimo3/configs/sft_qwen.yaml',
}

mine_cfg_path = mine_cfg_map[MODEL_KEY]
grpo_cfg_path = grpo_cfg_map[MODEL_KEY]
eval_cfg_path = eval_cfg_map[MODEL_KEY]
sft_cfg_path = sft_cfg_map[MODEL_KEY]

rl_train_out = f'{RL_TRAIN_ROOT}/mined_baseline_{MODEL_KEY}_{EXPERIMENT_NAME}_v1'
grpo_out = f'{MODELS_ROOT}/grpo_baseline_{MODEL_KEY}_{EXPERIMENT_NAME}'
eval_out_dir = f'{RUNS_ROOT}/evaluation_grpo_baseline_{MODEL_KEY}_{EXPERIMENT_NAME}'

print('mine config:', mine_cfg_path)
print('grpo config:', grpo_cfg_path)
print('eval config:', eval_cfg_path)
print('sft config:', sft_cfg_path)

In [ ]:
from aimo3.scripts.mine_baseline import mine_failures_baseline

with open(mine_cfg_path, 'r') as f:
    mine_cfg = yaml.safe_load(f)

mine_cfg['run']['experiment_name'] = EXPERIMENT_NAME
mine_cfg['run']['model_key'] = MODEL_KEY
mine_cfg['paths']['dataset_path'] = DATASET_PATH
mine_cfg['paths']['dataset_split'] = DATASET_SPLIT_FOR_MINING
mine_cfg['paths']['max_items'] = MAX_MINE_ITEMS
mine_cfg['paths']['output_path'] = rl_train_out
mine_cfg['reporting']['report_path'] = f'{RUNS_ROOT}/mining_report_baseline_{MODEL_KEY}_{EXPERIMENT_NAME}.json'

tmp_mine_cfg = f'/content/tmp_mine_baseline_{MODEL_KEY}_{EXPERIMENT_NAME}.yaml'
with open(tmp_mine_cfg, 'w') as f:
    yaml.safe_dump(mine_cfg, f, sort_keys=False)

print('Running mining with config:', tmp_mine_cfg)
mine_failures_baseline(tmp_mine_cfg)

In [ ]:
from aimo3.scripts.grpo_baseline import train_grpo_baseline

with open(grpo_cfg_path, 'r') as f:
    grpo_cfg = yaml.safe_load(f)

grpo_cfg['run']['experiment_name'] = EXPERIMENT_NAME
grpo_cfg['run']['model_key'] = MODEL_KEY
grpo_cfg['paths']['rl_train_path'] = rl_train_out
grpo_cfg['paths']['output_root'] = MODELS_ROOT
grpo_cfg['paths']['output_directory'] = grpo_out

tmp_grpo_cfg = f'/content/tmp_grpo_baseline_{MODEL_KEY}_{EXPERIMENT_NAME}.yaml'
with open(tmp_grpo_cfg, 'w') as f:
    yaml.safe_dump(grpo_cfg, f, sort_keys=False)

print('Running GRPO with config:', tmp_grpo_cfg)
train_grpo_baseline(tmp_grpo_cfg)

In [ ]:
from aimo3.scripts.evaluate import run_evaluation

# evaluate.py reads model_key and experiment_name from the SFT config path,
# so we write a temporary SFT config with the right run metadata.
with open(sft_cfg_path, 'r') as f:
    sft_cfg = yaml.safe_load(f)
sft_cfg['run']['experiment_name'] = EXPERIMENT_NAME
sft_cfg['run']['model_key'] = MODEL_KEY
tmp_sft_cfg = f'/content/tmp_sft_for_eval_{MODEL_KEY}_{EXPERIMENT_NAME}.yaml'
with open(tmp_sft_cfg, 'w') as f:
    yaml.safe_dump(sft_cfg, f, sort_keys=False)

with open(eval_cfg_path, 'r') as f:
    eval_cfg = yaml.safe_load(f)

eval_cfg['paths']['sft_config_path'] = tmp_sft_cfg
eval_cfg['paths']['dataset_path'] = DATASET_PATH
eval_cfg['paths']['model_checkpoint'] = grpo_out
eval_cfg['paths']['output_dir'] = eval_out_dir
eval_cfg['evaluation']['split'] = EVAL_SPLIT
eval_cfg['evaluation']['max_items'] = EVAL_MAX_ITEMS
eval_cfg['evaluation']['majn_max_items'] = EVAL_MAJN_ITEMS
eval_cfg['reporting']['save_report'] = True
eval_cfg['reporting']['report_filename'] = f'evaluation_report_grpo_baseline_{MODEL_KEY}_{EXPERIMENT_NAME}.json'

tmp_eval_cfg = f'/content/tmp_eval_grpo_baseline_{MODEL_KEY}_{EXPERIMENT_NAME}.yaml'
with open(tmp_eval_cfg, 'w') as f:
    yaml.safe_dump(eval_cfg, f, sort_keys=False)

print('Running evaluation with config:', tmp_eval_cfg)
results = run_evaluation(tmp_eval_cfg)
print(json.dumps(results, indent=2))